# Session 4: Convolutional Neural Networks and Filters

This notebook contains structured tasks based on Session 4 materials.

**Instructions:**
- Do NOT use a separate `.py` file. Complete all tasks directly in this notebook.
- Keep your code clear and well-commented.
- For Visulaization, please use `%matplotlib inline`
- Submit your completed notebook by uploading it to your forked repository.

### ***DO NOT UPLOAD THE DATASET TO GITHUB as it's too large***
<hr>

## Task 1: Convolutional Filters on an Image
Apply several 2D filters (Sobel X, Sobel Y, Laplacian, and Sharpen) to an image and visualize the results side by side.

**Hint:**
- Use OpenCV (`cv2.filter2D`) or NumPy to apply kernels.
- Define common kernels (Sobel, Laplacian, Sharpen).
- Use `matplotlib.pyplot` to plot the original and filtered images.

In [ ]:
%matplotlib inline

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cbook as cbook

# Use Matplotlib's built-in sample photograph so no external image download is needed.
with cbook.get_sample_data("grace_hopper.jpg") as image_file:
    image_rgb = plt.imread(image_file)

# Convert the RGB image to grayscale because the filters are applied to one channel.
gray_image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

# Define the convolution kernels.
sobel_x_kernel = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float32)

sobel_y_kernel = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.float32)

laplacian_kernel = np.array([
    [ 0,  1,  0],
    [ 1, -4,  1],
    [ 0,  1,  0]
], dtype=np.float32)

sharpen_kernel = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
], dtype=np.float32)

# Apply edge-detection filters using float output, then convert their absolute
# responses to the 8-bit range so they can be displayed clearly.
sobel_x = cv2.convertScaleAbs(cv2.filter2D(gray_image, cv2.CV_32F, sobel_x_kernel))
sobel_y = cv2.convertScaleAbs(cv2.filter2D(gray_image, cv2.CV_32F, sobel_y_kernel))
laplacian = cv2.convertScaleAbs(cv2.filter2D(gray_image, cv2.CV_32F, laplacian_kernel))

# The sharpened result is kept in the same 8-bit format as the input image.
sharpened = cv2.filter2D(gray_image, -1, sharpen_kernel)

# Display the original image and all four filtered results side by side.
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
results = [
    ("Original image", image_rgb, None),
    ("Grayscale image", gray_image, "gray"),
    ("Sobel X", sobel_x, "gray"),
    ("Sobel Y", sobel_y, "gray"),
    ("Laplacian", laplacian, "gray"),
    ("Sharpen", sharpened, "gray")
]

for ax, (title, image, cmap) in zip(axes.flat, results):
    ax.imshow(image, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()


## Task 2: Build a Simple CNN on FashionMNIST
Construct and train a small Convolutional Neural Network (CNN) on the FashionMNIST dataset.

**Hint:**
- Use `torchvision.datasets.FashionMNIST` for loading data.
- Define a small CNN with `nn.Conv2d`, `nn.ReLU`, `nn.MaxPool2d`, and `nn.Linear`.
- Train for a few epochs using an optimizer (e.g., Adam) and loss function (e.g., CrossEntropyLoss).
- Print training accuracy after each epoch.

*Note: FashionMNIST is downloaded automatically on the first run, so an internet connection is required.*


In [ ]:
# Import the libraries needed for the CNN and FashionMNIST dataset.
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Use a GPU when one is available; otherwise train on the CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(42)

# Convert each image to a tensor and normalize pixel values from [0, 1] to [-1, 1].
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# FashionMNIST is downloaded automatically the first time this cell is run.
data_root = "./data"
try:
    train_dataset = datasets.FashionMNIST(
        root=data_root, train=True, download=True, transform=transform
    )
    test_dataset = datasets.FashionMNIST(
        root=data_root, train=False, download=True, transform=transform
    )
except Exception as exc:
    raise RuntimeError(
        "FashionMNIST could not be downloaded. Check your internet connection "
        "and run this cell again. The dataset is intentionally not included in the notebook."
    ) from exc

# num_workers=0 keeps the notebook compatible with Windows and common notebook environments.
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

# A small CNN: convolution -> ReLU -> max pooling -> fully connected classifier.
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # 28x28 input becomes 14x14 after 2x2 max pooling.
        self.fc = nn.Linear(16 * 14 * 14, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc(x)
        return x

net = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

# Train for two epochs and report the accuracy after each epoch.
num_epochs = 2
for epoch in range(num_epochs):
    net.train()
    total_correct = 0
    total_samples = 0
    total_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = net(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        predictions = outputs.argmax(dim=1)
        total_correct += (predictions == labels).sum().item()
        total_samples += labels.size(0)

    train_accuracy = 100.0 * total_correct / total_samples
    average_loss = total_loss / total_samples
    print(
        f"Epoch [{epoch + 1}/{num_epochs}] - "
        f"Loss: {average_loss:.4f} - Training accuracy: {train_accuracy:.2f}%"
    )

# Evaluate the trained network on the test set as an additional check.
net.eval()
test_correct = 0
test_total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = net(images)
        test_correct += (outputs.argmax(dim=1) == labels).sum().item()
        test_total += labels.size(0)

print(f"Test accuracy: {100.0 * test_correct / test_total:.2f}%")


## Task 3: Visualize Feature Maps
Visualize feature maps produced by the first convolutional layer of your CNN.

**Hint:**
- Take a single image from the test set.
- Pass it through the first convolutional layer (`net.conv1`).
- Convert the output to NumPy and plot several channels as images.

In [ ]:
# Select one image from the FashionMNIST test dataset.
net.eval()
sample_image, sample_label = test_dataset[0]
sample_batch = sample_image.unsqueeze(0).to(device)  # Add the batch dimension.

# Pass the image through the first convolutional layer only.
with torch.no_grad():
    feature_maps = net.conv1(sample_batch).squeeze(0).cpu().numpy()

print("Image class:", test_dataset.classes[sample_label])
print("Feature-map array shape:", feature_maps.shape)

# Display the first 16 feature maps in a 4x4 grid.
num_maps = min(16, feature_maps.shape[0])
fig, axes = plt.subplots(4, 4, figsize=(10, 10))

for index, ax in enumerate(axes.flat):
    if index < num_maps:
        ax.imshow(feature_maps[index], cmap="gray")
        ax.set_title(f"Feature map {index + 1}")
    ax.axis("off")

plt.suptitle("Feature Maps from the First Convolutional Layer", y=1.02)
plt.tight_layout()
plt.show()


## Task 4: Haar Cascade Face Detection

Apply Haar Cascade classifiers for basic object detection (faces).

**Hint:**
- Use `cv2.CascadeClassifier` with a pre-trained XML file (e.g., `haarcascade_frontalface_default.xml`).
- Convert the input image to grayscale before detection.
- Use `detectMultiScale` to get bounding boxes.
- Draw rectangles on detected faces with `cv2.rectangle`.
- Visualize with Matplotlib.


In [ ]:
import cv2
# Reuse the photograph loaded in Task 1. Load it again if this cell is run on its own.
if "image_rgb" not in globals():
    import matplotlib.pyplot as plt
    import matplotlib.cbook as cbook
    with cbook.get_sample_data("grace_hopper.jpg") as image_file:
        image_rgb = plt.imread(image_file)

# Haar Cascade detection expects a grayscale image.
face_image_gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

# Load OpenCV's pre-trained frontal-face Haar Cascade classifier.
cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(cascade_path)
if face_cascade.empty():
    raise FileNotFoundError(f"Could not load Haar Cascade file: {cascade_path}")

# Detect faces. Each result contains (x, y, width, height).
faces = face_cascade.detectMultiScale(
    face_image_gray,
    scaleFactor=1.1,
    minNeighbors=5,
    minSize=(30, 30)
)
print(f"Number of faces detected: {len(faces)}")

# Draw a green rectangle around every detected face.
detected_image = image_rgb.copy()
for (x, y, width, height) in faces:
    cv2.rectangle(
        detected_image,
        (x, y),
        (x + width, y + height),
        (0, 255, 0),
        thickness=3
    )

# Compare the original photograph with the detection result.
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(image_rgb)
axes[0].set_title("Original image")
axes[0].axis("off")
axes[1].imshow(detected_image)
axes[1].set_title(f"Haar Cascade detections ({len(faces)} face(s))")
axes[1].axis("off")
plt.tight_layout()
plt.show()
